# RSNA original spatial attention: prospective training ablation

This notebook trains a new model. It does not copy a competition checkpoint or promise a leaderboard score.

The single hypothesis is **per-finding spatial attention**, compared with uniform spatial pooling. Both arms start from the same independently pretrained ConvNeXt backbone and shared MIL parameters. Zero query vectors reproduce the control output while remaining learnable. The ConvNeXt post-pool normalisation is retained.

A fresh deterministic split reserves 20% of report/known-relation groups as a locked holdout. Four development folds exclude all 58 expert-labelled studies and their known connected groups. Only development weak-label AUC selects checkpoints. The holdout is opened once, after every paired fold has completed; it measures agreement with weak labels, not expert accuracy. Known patient relations may be supplied to strengthen isolation.

**Attach these inputs before running, with T4 ×2 and Internet off:**

- Competition `rsna-knee-abnormality-detection` (`train.csv`).
- `dreaddevelopment/knee-raptor-corpus`: `all_vols.npy`, `all_masks.npy`, `all_ids.npy`.
- `dreaddevelopment/knee-raptor-corpus-ext`: `extra_vols.npy`, `extra_masks.npy`, `extra_ids.npy`.
- Dataset `rsna-knee-hybrid-report-labels`: `report_labels_v4hybrid.csv`.
- `stevenleehans/rsna-knee-llm-report-labels`: `llm_labels_v4_blend.csv`.
- Your existing output `tdeng24/rsna-knee-overnight-convnext-td`, containing the pure `imagenet_initialization.pt`. Required SHA256: `6270cae1e860a267e3b9a6f73932d627edaf3529a32bf9b80e79f04c936aa7b9`. No competition-trained weights are loaded.

The public caches contain 44 **sampled** planes at 336 pixels. A triplet never crosses a series slot, and all three masks must be positive. These are not claimed to be adjacent native DICOM slices. Source processing is the original Dread corpus44 contract; this experiment cannot serve a native384/96-plane checkpoint.

Default run: both arms of fold0, eight epochs each. If both complete, retain the entire output directory and extend the same locked plan with `MODE='all'`. `EVALUATE_HOLDOUT_ONCE=True` is valid only after all four pairs complete. Partial models have resume checkpoints and cannot count as a completed family. To resume across Kaggle sessions, attach the preceding output and set `RESUME_OUTPUT` to its `original_spatial` folder.

Only a checkpoint/receipt folder is generated; there is no competition submission in this training notebook. Do not publish reports, target tables, study IDs, or image arrays with the source repository.

Attribution: public corpus inputs by dreaddevelopment; public weak labels by their dataset authors. The per-finding spatial architecture, controlled comparison, split protocol, and training implementation in this notebook are original project code. All upstream input licences and competition rules still apply.


In [ ]:
import hashlib,json,os,shutil,subprocess,sys,time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable the T4 GPU accelerator'
assert torch.cuda.device_count()==2, 'Use T4 ×2 for paired arms'
CODE_ROOT=Path('/kaggle/working/original_spatial_code')
OUTPUT=Path('/kaggle/working/original_spatial')
MODE='paired-first'  # After the first complete pair, extend the same plan with 'all'.
EVALUATE_HOLDOUT_ONCE=False
RESUME_OUTPUT=''  # Optional attached previous original_spatial output directory.
RELATIONS_CSV=''  # Optional UID,RelatedStudyInstanceUID local table.
EXCLUDED_CSV=''   # Optional StudyInstanceUID local exclusion table.
if RESUME_OUTPUT:
    old=Path(RESUME_OUTPUT)
    assert (old/'plan.json').is_file(), 'Resume folder must contain the locked plan'
    shutil.copytree(old,OUTPUT,dirs_exist_ok=True)
OUTPUT.mkdir(parents=True,exist_ok=True)
os.environ['HF_HUB_OFFLINE']='1'
os.environ['PIP_NO_INDEX']='1'


In [ ]:
EMBEDDED={'src/rsna/spatial_mil.py': '"""Original per-finding spatial attention ablation over the public sampled-plane cache.\n\nThe uniform and spatial arms share an identical ConvNeXt encoder and target-wise\nslot/position MIL head. Spatial queries start at zero. Attention-minus-uniform\npooling keeps the initial output exactly uniform while retaining query gradients.\nThe actual backbone\'s forward_head preserves its pooling-then-normalisation order.\n"""\nfrom __future__ import annotations\nimport copy\nimport math\nimport os\nfrom pathlib import Path\nfrom typing import Any\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\',\n           \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\n\ndef preprocess_windows(windows: torch.Tensor, config: dict[str, Any], augment: bool = False) -> torch.Tensor:\n    """uint8 B×K×3×H×W -> ImageNet-normalised B×K×3×resolution×resolution.\n\n    Cache planes are sampled positions, not adjacent native DICOM slices.\n    Gain augmentation is shared across every window/channel of each study.\n    """\n    if windows.ndim != 5 or windows.shape[2] != 3 or windows.dtype != torch.uint8:\n        raise ValueError(\'Expected uint8 B×K×3×H×W windows\')\n    b, k = windows.shape[:2]\n    x = windows.float().div(255.)\n    if augment:\n        gain = torch.empty((b, 1, 1, 1, 1), device=x.device).uniform_(.9, 1.1)\n        x = (x * gain).clamp(0, 1)\n    res = int(config.get(\'resolution\', 320))\n    x = F.interpolate(x.flatten(0, 1), size=(res, res), mode=\'bilinear\', align_corners=False)\n    mean = x.new_tensor(config.get(\'mean\', [.485, .456, .406])).view(1, 3, 1, 1)\n    std = x.new_tensor(config.get(\'std\', [.229, .224, .225])).view(1, 3, 1, 1)\n    if (std <= 0).any():\n        raise ValueError(\'Normalisation standard deviations must be positive\')\n    return ((x - mean) / std).view(b, k, 3, res, res)\n\n\nclass SpatialTargetMIL(nn.Module):\n    """Twelve spatial readouts, then the same per-finding gated window MIL head.\n\n    A supplied backbone must expose num_features, forward_features(NCHW), and\n    forward_head(NCHW, pre_logits=True). This injection avoids external downloads\n    in CPU tests. The production backbone is a timm ConvNeXt with global_pool=\'avg\'.\n    """\n    def __init__(self, config: dict[str, Any], spatial: bool, *, backbone: nn.Module | None = None,\n                 pretrained: bool = False):\n        super().__init__()\n        self.config = copy.deepcopy(config)\n        self.spatial = bool(spatial)\n        if backbone is None:\n            import timm\n            backbone = timm.create_model(config.get(\'architecture\', \'convnext_tiny.fb_in22k_ft_in1k\'),\n                                         pretrained=pretrained, num_classes=0, global_pool=\'avg\', in_chans=3)\n        self.backbone = backbone\n        self.feature_dim = dim = int(backbone.num_features)\n        self.norm = nn.LayerNorm(dim)\n        self.slot = nn.Embedding(6, dim, padding_idx=5)\n        self.position = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, dim))\n        self.att_v = nn.Linear(dim, 256)\n        self.att_u = nn.Linear(dim, 256)\n        self.att_out = nn.Linear(256, len(TARGETS))\n        self.classifier_weight = nn.Parameter(torch.empty(len(TARGETS), dim))\n        self.classifier_bias = nn.Parameter(torch.zeros(len(TARGETS)))\n        self.dropout = nn.Dropout(float(config.get(\'head_dropout\', .20)))\n        nn.init.normal_(self.slot.weight, std=.01)\n        with torch.no_grad():\n            self.slot.weight[5].zero_()\n        nn.init.trunc_normal_(self.classifier_weight, std=.02)\n        # Create the extra spatial parameters last: ordinary MIL initialisation is unchanged.\n        key_dim = int(config.get(\'spatial_key_dim\', 64))\n        self.spatial_key = nn.Linear(dim, key_dim, bias=False)\n        self.spatial_queries = nn.Parameter(torch.zeros(len(TARGETS), key_dim))\n        if not spatial:\n            self.spatial_key.requires_grad_(False)\n            self.spatial_queries.requires_grad_(False)\n        if config.get(\'gradient_checkpointing\', False):\n            if not hasattr(backbone, \'set_grad_checkpointing\'):\n                raise ValueError(\'Backbone does not support gradient checkpointing\')\n            backbone.set_grad_checkpointing(True)\n\n    def pool_spatial_maps(self, maps: torch.Tensor) -> torch.Tensor:\n        """N×D×H×W -> N×12×D using the backbone\'s own post-pool head.\n\n        No normalisation is moved ahead of spatial pooling. Subtracting the\n        uniform attention makes zero-query initialisation exact without blocking\n        the derivative of attention with respect to the query parameters.\n        """\n        if maps.ndim != 4 or maps.shape[1] != self.feature_dim:\n            raise ValueError(\'Expected backbone spatial N×D×H×W map\')\n        n, d, height, width = maps.shape\n        base = F.adaptive_avg_pool2d(maps, 1).flatten(1)\n        pooled = base[:, None, :].expand(n, len(TARGETS), d)\n        if self.spatial:\n            tokens = maps.flatten(2).transpose(1, 2)\n            keys = self.spatial_key(F.layer_norm(tokens.float(), (d,)).to(tokens.dtype))\n            scores = torch.einsum(\'npd,cd->npc\', keys, self.spatial_queries.to(keys.dtype)) / math.sqrt(keys.shape[-1])\n            attention = scores.float().softmax(dim=1)\n            delta = attention - (1. / (height * width))\n            residual = torch.einsum(\'npc,npd->ncd\', delta.to(tokens.dtype), tokens)\n            pooled = pooled + residual\n        head_input = pooled.reshape(n * len(TARGETS), d, 1, 1).contiguous()\n        out = self.backbone.forward_head(head_input, pre_logits=True)\n        if out.shape != (n * len(TARGETS), d):\n            raise ValueError(\'Backbone forward_head changed its feature contract\')\n        return out.view(n, len(TARGETS), d)\n\n    def encode_windows(self, normalized_images: torch.Tensor, chunk: int = 0) -> torch.Tensor:\n        """Encode real windows only; never send padded windows through the encoder."""\n        if len(normalized_images) == 0:\n            raise ValueError(\'No valid windows to encode\')\n        step = chunk or len(normalized_images)\n        if step < 1:\n            raise ValueError(\'Chunk size must be positive\')\n        return torch.cat([self.pool_spatial_maps(self.backbone.forward_features(normalized_images[i:i + step]))\n                          for i in range(0, len(normalized_images), step)], dim=0)\n\n    def pool_features(self, features: torch.Tensor, slots: torch.Tensor, mask: torch.Tensor,\n                      positions: torch.Tensor) -> torch.Tensor:\n        """B×K×12×D -> B×12, with the inherited slot/position gated MIL formula."""\n        if features.ndim != 4 or features.shape[2:] != (len(TARGETS), self.feature_dim):\n            raise ValueError(\'Expected B×K×12×D class-specific window features\')\n        if slots.shape != features.shape[:2] or mask.shape != slots.shape or positions.shape != slots.shape:\n            raise ValueError(\'Bag metadata shape mismatch\')\n        valid = mask.bool()\n        if not valid.any(1).all():\n            raise ValueError(\'All-invalid study bag is not a prediction\')\n        if ((slots < 0) | (slots > 5)).any() or not torch.isfinite(positions).all():\n            raise ValueError(\'Invalid slot or position metadata\')\n        h = self.norm(features.float())\n        embeddings = self.slot(slots.long()) + self.position(positions.float().unsqueeze(-1))\n        h = h + embeddings[:, :, None, :]\n        # One dropout mask per window/feature, shared over class readouts. At\n        # uniform pooling this is the ordinary head\'s B×K×D dropout operation.\n        if self.training and self.dropout.p:\n            h = h * self.dropout(torch.ones_like(h[:, :, 0, :]))[:, :, None, :]\n        gated = torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))\n        scores = torch.einsum(\'bkcd,cd->bkc\', gated, self.att_out.weight) + self.att_out.bias\n        scores = scores.float().masked_fill(~valid[:, :, None], float(\'-inf\'))\n        attention = scores.softmax(dim=1)\n        pooled = torch.einsum(\'bkc,bkcd->bcd\', attention, h)\n        logits = (pooled * self.classifier_weight[None]).sum(-1) + self.classifier_bias\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError(\'Nonfinite study logits\')\n        return logits\n\n    def forward(self, windows: torch.Tensor, slots: torch.Tensor, mask: torch.Tensor,\n                positions: torch.Tensor, augment: bool = False, chunk: int = 0) -> torch.Tensor:\n        if mask.ndim != 2 or not mask.bool().any(1).all():\n            raise ValueError(\'All-invalid study bag is not a prediction\')\n        x = preprocess_windows(windows, self.config, augment)\n        b, k = x.shape[:2]\n        if mask.shape != (b, k):\n            raise ValueError(\'Window mask shape mismatch\')\n        valid_features = self.encode_windows(x[mask.bool()], chunk)\n        features = valid_features.new_zeros((b, k, len(TARGETS), self.feature_dim))\n        features[mask.bool()] = valid_features\n        return self.pool_features(features, slots, mask, positions)\n\n\ndef paired_models(config: dict[str, Any], *, backbone: nn.Module | None = None,\n                  pretrained: bool = False) -> tuple[SpatialTargetMIL, SpatialTargetMIL]:\n    """Identical initial bytes for both arms; only the spatial path is enabled."""\n    control = SpatialTargetMIL(config, False, backbone=backbone, pretrained=pretrained)\n    spatial = SpatialTargetMIL(config, True, backbone=copy.deepcopy(control.backbone), pretrained=False)\n    spatial.load_state_dict(control.state_dict(), strict=True)\n    return control, spatial\n\n\ndef save_checkpoint(model: SpatialTargetMIL, path: str | Path, *, contract: dict[str, Any],\n                    extras: dict[str, Any] | None = None) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    payload = {\'schema\': \'rsna_spatial_ablation_v1\', \'config\': model.config,\n               \'spatial\': model.spatial, \'contract\': contract,\n               \'state_dict\': {k: v.detach().cpu() for k, v in model.state_dict().items()}}\n    if extras:\n        if set(extras) & set(payload):\n            raise ValueError(\'Extras overwrite checkpoint contract\')\n        payload.update(extras)\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\ndef load_checkpoint(model: SpatialTargetMIL, path: str | Path, *, contract: dict[str, Any]) -> dict[str, Any]:\n    payload = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if payload.get(\'schema\') != \'rsna_spatial_ablation_v1\' or payload.get(\'config\') != model.config \\\n            or payload.get(\'spatial\') != model.spatial or payload.get(\'contract\') != contract:\n        raise ValueError(\'Checkpoint architecture, arm, configuration, or split contract differs\')\n    model.load_state_dict(payload[\'state_dict\'], strict=True)\n    return payload\n', 'src/rsna/cache_data.py': '"""Public corpus44 inputs, confidence-weighted targets, and a prospective split.\n\nActual inputs:\n  dreaddevelopment/knee-raptor-corpus: all_ids.npy, all_masks.npy, all_vols.npy\n  dreaddevelopment/knee-raptor-corpus-ext: extra_ids.npy, extra_masks.npy, extra_vols.npy\nThe uint8 volumes have 44 sampled planes at 336 pixels. Neighbouring cached\nplanes need not be neighbouring native DICOM slices. Triplets never cross slots.\nOnly generated local experiment directories contain IDs or label arrays.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport os\nimport re\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\',\n           \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nUID = \'StudyInstanceUID\'\nSLOT_WIDTHS = (12, 10, 8, 6, 8)\nCONTRACT = {\'name\': \'dread_corpus44_slot_safe_spatial_v1\', \'cache_shape\': [44, 336, 336],\n            \'cache_dtype\': \'uint8\', \'slot_widths\': list(SLOT_WIDTHS),\n            \'span\': [.15, .85], \'crop_mm\': 140., \'percentiles\': [2., 98.],\n            \'cache_resize\': \'cv2.INTER_AREA\', \'uint8_conversion\': \'multiply_255_then_truncate\',\n            \'triplets\': \'three positive-mask sampled planes inside one slot\', \'max_windows\': 34,\n            \'positions\': \'2*(center-slot_start)/(slot_width-1)-1\', \'padding_slot\': 5,\n            \'native_slice_adjacency_established\': False}\nDEFAULT_CONFIG = {\'architecture\': \'convnext_tiny.fb_in22k_ft_in1k\', \'resolution\': 320,\n                  \'mean\': [.485, .456, .406], \'std\': [.229, .224, .225],\n                  \'gradient_checkpointing\': True, \'head_dropout\': .2, \'spatial_key_dim\': 64,\n                  \'seed\': 20261009, \'n_folds\': 4, \'holdout_fraction\': .20, \'epochs\': 8,\n                  \'train_windows\': 24, \'batch_size\': 2, \'eval_batch_size\': 2, \'workers\': 2,\n                  \'backbone_lr\': 3e-5, \'head_lr\': 1e-3, \'weight_decay\': .02,\n                  \'positive_weight_cap\': 5., \'label_smoothing\': .02, \'gradient_clip\': 3.,\n                  \'max_hours\': 8.5, \'reserve_minutes\': 20,\n                  \'initialization_sha256\': \'6270cae1e860a267e3b9a6f73932d627edaf3529a32bf9b80e79f04c936aa7b9\'}\nDEFAULT_TEACHERS = [(\'rsna-knee-hybrid-report-labels\', \'report_labels_v4hybrid.csv\'),\n                    (\'rsna-knee-llm-report-labels\', \'llm_labels_v4_blend.csv\')]\n\n\ndef sha256(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(8 << 20), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\n\ndef volume_fingerprint(path, full=False):\n    """Record exact size and either a full or explicitly sampled content digest.\n\n    Full hashing is opt-in: these public arrays total roughly22GB. Sampled\n    hashing checks the header and16evenly-spaced64KiBblocks; it is not a proof\n    that every pixel byte is unchanged. Immutable Kaggle dataset inputs should\n    be reused; callers may enable full hashing for stronger local provenance.\n    """\n    path=Path(path)\n    if full:\n        return {\'bytes\':path.stat().st_size,\'method\':\'full_sha256\',\'sha256\':sha256(path)}\n    size=path.stat().st_size\n    h=hashlib.sha256()\n    offsets=np.linspace(0,max(0,size-65536),16).round().astype(np.int64)\n    with path.open(\'rb\') as f:\n        for offset in offsets:\n            f.seek(int(offset)); h.update(str(int(offset)).encode()); h.update(f.read(65536))\n    return {\'bytes\':size,\'method\':\'sampled_16x64KiB_sha256_not_full_pixel_verification\',\'sha256\':h.hexdigest()}\n\ndef atomic_json(payload, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    temporary.write_text(json.dumps(payload, indent=2))\n    os.replace(temporary, path)\n\n\ndef find_dataset_file(slug: str, filename: str, input_root=\'/kaggle/input\') -> str:\n    root = Path(input_root)\n    candidates = [root / slug] + list(root.glob(\'datasets/*/\' + slug)) + list(root.glob(\'notebooks/*/\' + slug))\n    hits = sorted({str(p.resolve()) for d in candidates if d.is_dir() for p in d.rglob(filename) if p.is_file()})\n    if len(hits) != 1:\n        raise FileNotFoundError(f\'Expected one {slug}/{filename}; found {hits}\')\n    return hits[0]\n\n\ndef discover_inputs(input_root=\'/kaggle/input\'):\n    paths = {}\n    for suffix, slug, prefix in [(\'a\', \'knee-raptor-corpus\', \'all\'), (\'b\', \'knee-raptor-corpus-ext\', \'extra\')]:\n        for kind in (\'ids\', \'masks\', \'vols\'):\n            paths[kind + \'_\' + suffix] = find_dataset_file(slug, prefix + \'_\' + kind + \'.npy\', input_root)\n    root = Path(input_root)\n    train = [p for p in (root / \'rsna-knee-abnormality-detection\' / \'train.csv\',\n                        root / \'competitions\' / \'rsna-knee-abnormality-detection\' / \'train.csv\') if p.is_file()]\n    if len(train) != 1:\n        raise FileNotFoundError(\'Attach competition train.csv exactly once\')\n    teachers = [find_dataset_file(s, f, input_root) for s, f in DEFAULT_TEACHERS]\n    return paths, str(train[0]), teachers\n\n\nclass Corpus:\n    def __init__(self, paths: dict[str, str], expected_counts=(3200, 1207)):\n        self.paths = dict(paths)\n        self.expected_counts = expected_counts\n        self.arrays = [np.load(paths[\'vols_\' + s], mmap_mode=\'r\', allow_pickle=False) for s in (\'a\', \'b\')]\n        ids = [np.load(paths[\'ids_\' + s], allow_pickle=False).astype(str) for s in (\'a\', \'b\')]\n        masks = [np.load(paths[\'masks_\' + s], allow_pickle=False) for s in (\'a\', \'b\')]\n        if expected_counts is not None and tuple(len(a) for a in self.arrays) != tuple(expected_counts):\n            raise ValueError(\'Wrong public corpus sizes; expected 3200 + 1207 studies\')\n        for volume, uid, mask in zip(self.arrays, ids, masks):\n            if volume.shape != (len(uid), 44, 336, 336) or volume.dtype != np.uint8:\n                raise ValueError(\'Expected uint8 study volumes N×44×336×336\')\n            if mask.shape != (len(uid), 44) or mask.dtype != np.uint8 or not np.isin(mask, [0, 1]).all():\n                raise ValueError(\'Expected binary uint8 N×44 masks aligned to the same IDs\')\n        self.ids, self.masks = np.concatenate(ids), np.concatenate(masks)\n        if len(set(self.ids)) != len(self.ids):\n            raise ValueError(\'Duplicate corpus UIDs; never independently sort arrays\')\n        self.boundary = len(self.arrays[0])\n    def __len__(self):\n        return len(self.ids)\n    def __getitem__(self, row):\n        return self.arrays[0][row] if row < self.boundary else self.arrays[1][row - self.boundary]\n    def __getstate__(self):\n        return self.paths, self.expected_counts\n    def __setstate__(self, state):\n        self.__init__(*state)\n\n\ndef eligible_centers(mask):\n    mask = np.asarray(mask)\n    if mask.shape != (44,) or not np.isin(mask, [0, 1]).all():\n        raise ValueError(\'Expected a binary 44-plane corpus mask\')\n    rows, offset = [], 0\n    for slot, width in enumerate(SLOT_WIDTHS):\n        for center in range(offset + 1, offset + width - 1):\n            if mask[center - 1:center + 2].all():\n                rows.append((center, slot, 2 * (center - offset) / (width - 1) - 1))\n        offset += width\n    return np.asarray(rows, np.float64).reshape(-1, 3)\n\n\nclass Bags(Dataset):\n    def __init__(self, corpus, rows, targets, weights, config, *, train, fold):\n        self.corpus, self.rows = corpus, np.asarray(rows, dtype=np.int64)\n        self.targets, self.weights, self.config = targets, weights, config\n        self.train, self.fold, self.epoch = train, fold, 0\n        self.k = int(config[\'train_windows\']) if train else 34\n    def __len__(self):\n        return len(self.rows)\n    def __getitem__(self, index):\n        row = int(self.rows[index])\n        table = eligible_centers(self.corpus.masks[row])\n        if not len(table):\n            raise ValueError(f\'All-invalid cached study at row {row}\')\n        if self.train and len(table) > self.k:\n            rng = np.random.default_rng(self.config[\'seed\'] + self.fold * 1000003 + self.epoch * 10007 + row)\n            first = [int(rng.choice(np.flatnonzero(table[:, 1] == s))) for s in np.unique(table[:, 1])]\n            if len(first) > self.k:\n                raise ValueError(\'train_windows must cover each available slot\')\n            remaining = np.setdiff1d(np.arange(len(table)), first)\n            take = first + rng.choice(remaining, self.k - len(first), replace=False).tolist()\n            table = table[np.sort(take)]\n        n = min(len(table), self.k)\n        windows = np.zeros((self.k, 3, 336, 336), np.uint8)\n        slots, positions, mask = np.full(self.k, 5, np.int64), np.zeros(self.k, np.float32), np.zeros(self.k, bool)\n        centers = table[:n, 0].astype(np.int64)\n        windows[:n] = self.corpus[row][centers[:, None] + np.array([-1, 0, 1])]\n        slots[:n], positions[:n], mask[:n] = table[:n, 1].astype(np.int64), table[:n, 2], True\n        return (torch.from_numpy(windows), torch.from_numpy(slots), torch.from_numpy(mask),\n                torch.from_numpy(positions), torch.from_numpy(self.targets[row]),\n                torch.from_numpy(self.weights[row]), row)\n\n\ndef group_split(ids, reports, gold, *, seed, n_folds, holdout_fraction, relations=(), excluded_ids=()):\n    """Split report/known-relation components; gold-connected components are excluded.\n\n    Optional relation pairs should include known patient/study links. Without\n    these, report groups do not establish patient or scanner isolation.\n    """\n    if not (0 < holdout_fraction < .5) or n_folds < 2:\n        raise ValueError(\'Invalid holdout fraction or fold count\')\n    ids = np.asarray(ids).astype(str)\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Duplicate IDs\')\n    reports = [re.sub(r\'\\s+\', \' \', str(s).lower()).strip() for s in reports]\n    parent = {u: u for u in ids}\n    def find(u):\n        while parent[u] != u:\n            parent[u] = parent[parent[u]]\n            u = parent[u]\n        return u\n    def union(a, b):\n        a, b = find(a), find(b)\n        if a != b:\n            parent[max(a, b)] = min(a, b)\n    seen = {}\n    for uid, report in zip(ids, reports):\n        if report:\n            if report in seen:\n                union(uid, seen[report])\n            else:\n                seen[report] = uid\n    for a, b in relations:\n        a, b = str(a), str(b)\n        if a not in parent or b not in parent:\n            raise ValueError(\'Relation references UID outside the supplied training corpus\')\n        union(a, b)\n    components = {}\n    for uid in ids:\n        components.setdefault(find(uid), []).append(uid)\n    group_hash = {root: hashlib.sha256(\'\\n\'.join(sorted(members)).encode()).hexdigest()\n                  for root, members in components.items()}\n    groups = np.asarray([group_hash[find(u)] for u in ids])\n    excluded_ids = set(map(str, excluded_ids))\n    if excluded_ids - set(ids):\n        raise ValueError(\'Exclusion list references unknown corpus UID\')\n    forbidden = {groups[i] for i, u in enumerate(ids) if gold[i] or u in excluded_ids or not reports[i]}\n    roles, folds = [], []\n    for group in groups:\n        digest = hashlib.sha256(f\'holdout:{seed}:{group}\'.encode()).hexdigest()\n        if group in forbidden:\n            role, fold = \'excluded\', -1\n        elif int(digest[:16], 16) / 2**64 < holdout_fraction:\n            role, fold = \'holdout\', -1\n        else:\n            role = \'development\'\n            fold = int(hashlib.sha256(f\'fold:{seed}:{group}\'.encode()).hexdigest()[:16], 16) % n_folds\n        roles.append(role); folds.append(fold)\n    return np.asarray(groups), np.asarray(roles), np.asarray(folds, np.int16)\n\n\ndef make_plan(config, out_dir, *, paths, train_csv, teacher_csvs, relations_csv=None, excluded_csv=None):\n    """Lock the split before training. Existing plans are verified, never overwritten."""\n    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)\n    corpus = Corpus(paths)\n    train = pd.read_csv(train_csv, dtype={UID: str})\n    if train[UID].duplicated().any() or set(train[UID]) != set(corpus.ids):\n        raise ValueError(\'Official train table and cache UIDs differ\')\n    train = train.set_index(UID).loc[corpus.ids]\n    observed = train[TARGETS].notna().sum(1).to_numpy()\n    if not np.isin(observed, [0, len(TARGETS)]).all():\n        raise ValueError(\'Official expert row is partially labelled\')\n    gold = observed == len(TARGETS)\n    if gold.sum() != 58:\n        raise ValueError(\'Expected exactly 58 official expert-labelled studies\')\n    reports = train[\'Report\'].fillna(\'\').astype(str).tolist()\n    relations = []\n    if relations_csv:\n        frame = pd.read_csv(relations_csv, dtype=str)\n        relations = list(zip(frame[UID], frame[\'RelatedStudyInstanceUID\']))\n    excluded = []\n    if excluded_csv:\n        excluded = pd.read_csv(excluded_csv, usecols=[UID], dtype=str)[UID].tolist()\n    groups, roles, folds = group_split(corpus.ids, reports, gold, seed=config[\'seed\'], n_folds=config[\'n_folds\'],\n                                      holdout_fraction=config[\'holdout_fraction\'], relations=relations, excluded_ids=excluded)\n    sources, confidence, label_receipts = [], [], []\n    if not teacher_csvs:\n        raise ValueError(\'At least one weak-label source is required\')\n    for path in teacher_csvs:\n        frame = pd.read_csv(path, dtype={UID: str})\n        if frame[UID].duplicated().any():\n            raise ValueError(\'Duplicate weak-label source UID\')\n        frame = frame.set_index(UID).reindex(corpus.ids)\n        values = frame[TARGETS].to_numpy(np.float32)\n        known = np.isfinite(values)\n        if ((values[known] < 0) | (values[known] > 1)).any():\n            raise ValueError(\'Weak probabilities must be in [0,1]\')\n        sources.append(values)\n        cols = [t + \'__conf\' for t in TARGETS]\n        c = frame[cols].to_numpy(np.float32) if all(k in frame for k in cols) else np.full_like(values, .75)\n        c = np.nan_to_num(c, nan=.25)\n        if ((c < 0) | (c > 1)).any():\n            raise ValueError(\'Weak-label confidence must be in [0,1]\')\n        confidence.append(np.clip(c, .25, 1.))\n        label_receipts.append({\'path\':str(path),\'sha256\':sha256(path)})\n    cube = np.stack(sources)\n    count = np.isfinite(cube).sum(0)\n    y = np.nansum(cube, axis=0) / np.maximum(count, 1)\n    disagreement = np.nansum(np.abs(cube - y[None]), axis=0) / np.maximum(count, 1)\n    weights = (np.clip(1 - 2 * disagreement, .25, 1) * np.mean(confidence, axis=0)).astype(np.float32)\n    weights[count == 0] = 0\n    usable = np.array([len(eligible_centers(m)) > 0 for m in corpus.masks])\n    roles[~usable | ~(count > 0).all(1)] = \'excluded\'\n    folds[roles != \'development\'] = -1\n    y[gold], weights[gold] = .5, 0\n    if not (roles == \'holdout\').any() or not (roles == \'development\').any():\n        raise ValueError(\'Hash split produced an empty holdout/development partition\')\n    for fold in range(config[\'n_folds\']):\n        tr = (roles == \'development\') & (folds != fold)\n        va = (roles == \'development\') & (folds == fold)\n        if not tr.any() or not va.any():\n            raise ValueError(\'Hash split produced an empty training/validation fold\')\n        if set(groups[tr]) & set(groups[va]) or set(groups[tr | va]) & set(groups[gold | (roles == \'holdout\')]):\n            raise ValueError(\'Group isolation violation\')\n        for j, target in enumerate(TARGETS):\n            hard = y[va, j][np.abs(y[va, j] - .5) > 1e-6] > .5\n            if hard.sum() < 2 or (~hard).sum() < 2:\n                raise ValueError(f\'Fixed fold {fold} lacks both weak-label classes for {target}; do not re-roll seed after inspection\')\n    inputs = {\'train_csv\':{\'path\':str(train_csv),\'sha256\':sha256(train_csv)},\'teachers\':label_receipts,\n              \'alignment\':{k:sha256(p) for k,p in paths.items() if not k.startswith(\'vols\')},\n              \'volumes\':{k:volume_fingerprint(p,config.get(\'full_volume_sha256\',False)) for k,p in paths.items() if k.startswith(\'vols\')},\n              \'relations\':None if not relations_csv else sha256(relations_csv),\n              \'exclusions\':None if not excluded_csv else sha256(excluded_csv)}\n    source_paths = [Path(__file__), Path(__file__).with_name(\'spatial_mil.py\')]\n    trainer = Path(__file__).parents[2] / \'scripts\' / \'train_spatial_ablation.py\'\n    if trainer.is_file():\n        source_paths.append(trainer)\n    source_hashes = {p.name:sha256(p) for p in source_paths}\n    lock = out / \'plan.json\'\n    if lock.exists():\n        plan = json.loads(lock.read_text())\n        if plan[\'config\'] != config or plan[\'inputs\'] != inputs or plan[\'source_sha256\'] != source_hashes:\n            raise ValueError(\'Existing locked plan differs from inputs, source, or configuration\')\n        if sha256(out/\'split_manifest.csv\') != plan[\'split_manifest_sha256\'] or sha256(out/\'targets.npz\') != plan[\'targets_sha256\']:\n            raise ValueError(\'Locked split or targets changed\')\n        return plan\n    manifest = pd.DataFrame({UID:corpus.ids,\'corpus_row\':np.arange(len(corpus)),\'group_sha256\':groups,\n                             \'role\':roles,\'fold\':folds,\'is_gold\':gold,\'usable_pixels\':usable})\n    manifest.to_csv(out/\'split_manifest.csv\', index=False)\n    np.savez_compressed(out/\'targets.npz\', ids=corpus.ids, targets=y.astype(np.float32), weights=weights,\n                        roles=roles, folds=folds, groups=groups, gold=gold)\n    plan = {\'schema\':\'rsna_spatial_ablation_plan_v1\',\'config\':config,\'pixel_contract\':CONTRACT,\n            \'corpus_paths\':paths,\'inputs\':inputs,\'source_sha256\':source_hashes,\n            \'split_manifest_sha256\':sha256(out/\'split_manifest.csv\'),\'targets_sha256\':sha256(out/\'targets.npz\'),\n            \'counts\':{k:int((roles==k).sum()) for k in (\'development\',\'holdout\',\'excluded\')},\n            \'gold_rows\':int(gold.sum()),\'gold_gradient_or_selection_rows\':0,\n            \'validation_boundary\':\'Fresh report/known-relation components. No patient/scanner isolation is established without relation inputs. Holdout never selects checkpoints; weak labels are a proxy.\'}\n    atomic_json(plan, lock)\n    return plan\n', 'scripts/train_spatial_ablation.py': '#!/usr/bin/env python3\n"""Train a prospective, paired uniform-vs-spatial MIL experiment, offline.\n\nT4x2: one independent arm per GPU, with identical initialisation, UID split,\nwindow samples, sampler order, epoch RNG seed and optimisation schedule. The\nfirst pair runs fold0 before the remaining folds. Partial models are retained\nfor exact resume, but never counted as complete or used for the final holdout.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\nfrom sklearn.metrics import roc_auc_score\nfrom rsna.cache_data import (Bags, Corpus, TARGETS, UID, CONTRACT, DEFAULT_CONFIG, atomic_json,\n                             discover_inputs, make_plan, sha256, volume_fingerprint)\nfrom rsna.spatial_mil import SpatialTargetMIL, load_checkpoint, save_checkpoint\n\n\ndef versions():\n    return {\'python\':platform.python_version(),\n            **{p:importlib.metadata.version(p) for p in (\'torch\',\'timm\',\'numpy\',\'pandas\',\'scikit-learn\')},\n            \'cuda\':str(torch.version.cuda)}\n\n\ndef weak_scores(truth, pred):\n    scores={}\n    for j,target in enumerate(TARGETS):\n        known=np.isfinite(truth[:,j]) & (np.abs(truth[:,j]-.5)>1e-6)\n        binary=(truth[known,j]>.5).astype(np.uint8)\n        if len(np.unique(binary))<2:\n            raise ValueError(f\'Both weak-label classes absent for {target}\')\n        scores[target]=float(roc_auc_score(binary,pred[known,j]))\n    return float(np.mean(list(scores.values()))),scores\n\n\ndef source_paths():\n    base=Path(__file__).resolve().parents[1]\n    return {\'train_spatial_ablation.py\':Path(__file__).resolve(),\n            \'cache_data.py\':base/\'src/rsna/cache_data.py\',\'spatial_mil.py\':base/\'src/rsna/spatial_mil.py\'}\n\n\ndef verify_plan(plan_path):\n    plan_path=Path(plan_path)\n    plan=json.loads(plan_path.read_text())\n    if plan[\'pixel_contract\']!=CONTRACT:\n        raise ValueError(\'Pixel contract changed\')\n    for name,path in source_paths().items():\n        if plan[\'source_sha256\'].get(name)!=sha256(path):\n            raise ValueError(f\'Source changed after split lock: {name}\')\n    for name,key in [(\'split_manifest.csv\',\'split_manifest_sha256\'),(\'targets.npz\',\'targets_sha256\')]:\n        if sha256(plan_path.parent/name)!=plan[key]:\n            raise ValueError(\'Split or target table changed after lock\')\n    for name,digest in plan[\'inputs\'][\'alignment\'].items():\n        if sha256(plan[\'corpus_paths\'][name])!=digest:\n            raise ValueError(f\'Cache alignment changed: {name}\')\n    for name,fingerprint in plan[\'inputs\'][\'volumes\'].items():\n        if volume_fingerprint(plan[\'corpus_paths\'][name],fingerprint[\'method\']==\'full_sha256\')!=fingerprint:\n            raise ValueError(f\'Cache volume fingerprint changed: {name}\')\n    return plan\n\n\ndef initialized_model(config, arm, init_path, seed):\n    if sha256(init_path)!=config[\'initialization_sha256\']:\n        raise ValueError(\'Independent ImageNet initialisation SHA256 differs from locked configuration\')\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    model=SpatialTargetMIL(config, spatial=arm==\'spatial\', pretrained=False)\n    initial=torch.load(init_path,map_location=\'cpu\',weights_only=True)\n    if initial.get(\'architecture\')!=config[\'architecture\'] or \'state_dict\' not in initial:\n        raise ValueError(\'Expected the offline pure ImageNet backbone payload, not a trained competition model\')\n    pretrained=initial.get(\'pretrained_config\',{})\n    if list(pretrained.get(\'mean\',[]))!=config[\'mean\'] or list(pretrained.get(\'std\',[]))!=config[\'std\']:\n        raise ValueError(\'Initial backbone normalisation differs from the fixed image contract\')\n    model.backbone.load_state_dict(initial[\'state_dict\'],strict=True)\n    return model\n\n\ndef make_loader(dataset, config, train, fold, epoch):\n    generator=torch.Generator().manual_seed(config[\'seed\']+fold*997+epoch*10007)\n    return DataLoader(dataset,batch_size=config[\'batch_size\'] if train else config[\'eval_batch_size\'],\n                      shuffle=train,num_workers=config[\'workers\'],pin_memory=True,drop_last=False,\n                      persistent_workers=False,prefetch_factor=2 if config[\'workers\'] else None,\n                      generator=generator)\n\n\n@torch.inference_mode()\ndef predict(model, corpus, rows, values, config, fold, device):\n    dataset=Bags(corpus,rows,values[\'targets\'],values[\'weights\'],config,train=False,fold=fold)\n    loader=make_loader(dataset,config,False,fold,0)\n    model.eval(); indices=[]; probabilities=[]\n    for x,s,m,p,_,_,r in loader:\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            z=model(x.to(device,non_blocking=True),s.to(device),m.to(device),p.to(device),chunk=32)\n        prob=z.float().sigmoid().cpu().numpy()\n        if not np.isfinite(prob).all():\n            raise FloatingPointError(\'Nonfinite evaluation scores\')\n        indices.extend(r.tolist()); probabilities.append(prob)\n    ix=np.asarray(indices,np.int64)\n    if len(ix)!=len(rows) or len(set(ix))!=len(rows) or set(ix)!=set(rows):\n        raise ValueError(\'Evaluation UID coverage failure\')\n    return ix,np.concatenate(probabilities)\n\n\ndef train_arm(args):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Actual training requires CUDA; CPU tests do not train the model\')\n    torch.cuda.set_device(0); device=torch.device(\'cuda\',0)\n    plan=verify_plan(args.plan); config=plan[\'config\']; fold=args.fold; arm=args.arm\n    corpus=Corpus(plan[\'corpus_paths\'])\n    values=np.load(Path(args.plan).parent/\'targets.npz\',allow_pickle=False)\n    if not np.array_equal(values[\'ids\'],corpus.ids):\n        raise ValueError(\'Cache/target UID order differs\')\n    roles,folds,groups,gold=values[\'roles\'],values[\'folds\'],values[\'groups\'],values[\'gold\']\n    tr=np.flatnonzero((roles==\'development\') & (folds!=fold))\n    va=np.flatnonzero((roles==\'development\') & (folds==fold))\n    held=np.flatnonzero(roles==\'holdout\')\n    if not len(tr) or not len(va) or gold[tr].any() or gold[va].any() \\\n        or set(groups[tr]) & set(groups[va]) or set(groups[tr]) & set(groups[held]):\n        raise ValueError(\'Gold/holdout/group isolation violation\')\n    out=Path(args.out); out.mkdir(parents=True,exist_ok=True)\n    prefix=f\'{arm}_fold{fold}\'\n    status_path=out/(prefix+\'_status.json\')\n    seed=config[\'seed\']+fold*997\n    model=initialized_model(config,arm,args.init,seed).to(device)\n    # Packages are part of the resume contract; RNG streams cannot be compared\n    # across an silently changed torch/timm training implementation.\n    contract={\'arm\':arm,\'fold\':fold,\'config\':config,\'pixel_contract\':CONTRACT,\n              \'split_manifest_sha256\':plan[\'split_manifest_sha256\'],\'targets_sha256\':plan[\'targets_sha256\'],\n              \'source_sha256\':plan[\'source_sha256\'],\'initialization_sha256\':config[\'initialization_sha256\'],\n              \'train_uid_sha256\':hashlib.sha256(\'\\n\'.join(corpus.ids[tr]).encode()).hexdigest(),\n              \'validation_uid_sha256\':hashlib.sha256(\'\\n\'.join(corpus.ids[va]).encode()).hexdigest(),\n              \'packages\':versions(),\'gold_gradient_or_selection_rows\':0,\'holdout_gradient_or_selection_rows\':0,\n              \'selection\':\'development weak-label macro-AUC; holdout never selects an epoch\'}\n    if status_path.exists():\n        old=json.loads(status_path.read_text())\n        if old.get(\'contract\')!=contract:\n            raise ValueError(\'Existing arm status differs from locked experiment contract\')\n        if old[\'status\']==\'complete\':\n            load_checkpoint(model,out/(prefix+\'_best.pt\'),contract=contract)\n            print(\'ALREADY_COMPLETE\',prefix,flush=True); return\n    bb=[p for p in model.backbone.parameters() if p.requires_grad]\n    head=[p for n,p in model.named_parameters() if not n.startswith(\'backbone.\') and p.requires_grad]\n    optimizer=torch.optim.AdamW([{\'params\':bb,\'lr\':config[\'backbone_lr\']},\n                                {\'params\':head,\'lr\':config[\'head_lr\']}],weight_decay=config[\'weight_decay\'])\n    dataset=Bags(corpus,tr,values[\'targets\'],values[\'weights\'],config,train=True,fold=fold)\n    steps_per_epoch=math.ceil(len(dataset)/config[\'batch_size\'])\n    total_steps=steps_per_epoch*config[\'epochs\']\n    def schedule(step):\n        warmup=max(1,round(total_steps*.15))\n        if step<warmup:\n            return (step+1)/warmup\n        phase=(step-warmup)/max(1,total_steps-warmup)\n        return .05+.95*(1+math.cos(math.pi*min(1,phase)))/2\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,schedule)\n    scaler=torch.amp.GradScaler(\'cuda\')\n    y,w=values[\'targets\'],values[\'weights\']\n    prevalence=(y[tr]*w[tr]).sum(0)/w[tr].sum(0).clip(1e-6)\n    positive=torch.tensor(np.clip((1-prevalence)/prevalence.clip(.02),1,config[\'positive_weight_cap\']),device=device)\n    epoch=next_batch=0; best=-1.; history=[]; accumulated={\'loss\':0.,\'batches\':0,\'overflows\':0,\'updates\':0}\n    latest=out/(prefix+\'_latest.pt\')\n    def checkpoint(epoch_value,batch_value):\n        save_checkpoint(model,latest,contract=contract,extras={\n            \'epoch\':epoch_value,\'next_batch\':batch_value,\'best_score\':best,\'history\':history,\n            \'optimizer\':optimizer.state_dict(),\'scheduler\':scheduler.state_dict(),\'scaler\':scaler.state_dict(),\n            \'rng_cpu\':torch.get_rng_state(),\'rng_cuda\':torch.cuda.get_rng_state(device).cpu(),\n            \'accumulated\':accumulated,\'train_prevalence\':prevalence.tolist()})\n    if latest.exists():\n        resume=load_checkpoint(model,latest,contract=contract)\n        optimizer.load_state_dict(resume[\'optimizer\']); scheduler.load_state_dict(resume[\'scheduler\']); scaler.load_state_dict(resume[\'scaler\'])\n        epoch=int(resume[\'epoch\']); next_batch=int(resume[\'next_batch\']); best=float(resume[\'best_score\']); history=resume[\'history\']\n        accumulated=resume[\'accumulated\']; torch.set_rng_state(resume[\'rng_cpu\']); torch.cuda.set_rng_state(resume[\'rng_cuda\'],device)\n    else:\n        # Reset after model construction so paired arms begin identical RNG streams.\n        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n    # Recover the narrow crash window after the last latest checkpoint was\n    # written but before the final status JSON was committed.\n    if epoch >= config[\'epochs\']:\n        if len(history) != config[\'epochs\'] or not (out/(prefix+\'_best.pt\')).is_file():\n            raise ValueError(\'Completed checkpoint lacks its epoch history or selected model\')\n        load_checkpoint(model,out/(prefix+\'_best.pt\'),contract=contract)\n        atomic_json({\'status\':\'complete\',\'contract\':contract,\'completed_epochs\':epoch,\n                     \'best_weak_auc\':best,\'history\':history,\'recovered_final_status\':True},status_path)\n        return\n    torch.backends.cudnn.benchmark=False\n    started=time.time(); reserve=config[\'reserve_minutes\']*60\n    atomic_json({\'status\':\'running\',\'contract\':contract,\'completed_epochs\':epoch,\'history\':history},status_path)\n    while epoch<config[\'epochs\']:\n        if time.time()>args.deadline-reserve:\n            checkpoint(epoch,next_batch)\n            atomic_json({\'status\':\'partial\',\'contract\':contract,\'completed_epochs\':epoch,\'next_batch\':next_batch,\'history\':history,\n                         \'reason\':\'GPU budget stopped; resumable; not deployable\'},status_path)\n            return\n        dataset.epoch=epoch\n        loader=make_loader(dataset,config,True,fold,epoch)\n        model.train(); epoch_started=time.time(); consecutive_overflows=0\n        for batch_index,(x,s,m,p,target,weight,_) in enumerate(loader):\n            if batch_index<next_batch:\n                continue\n            # A mid-epoch resume skips deterministic loader batches without\n            # consuming augmentation/dropout RNG; the checkpoint stores that RNG.\n            if time.time()>args.deadline-reserve:\n                checkpoint(epoch,batch_index)\n                atomic_json({\'status\':\'partial\',\'contract\':contract,\'completed_epochs\':epoch,\'next_batch\':batch_index,\'history\':history,\n                             \'reason\':\'GPU budget stopped mid epoch; resumable; not deployable\'},status_path)\n                return\n            target,weight=target.to(device),weight.to(device)\n            smoothed=target*(1-config[\'label_smoothing\'])+config[\'label_smoothing\']/2\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                logits=model(x.to(device,non_blocking=True),s.to(device),m.to(device),p.to(device),augment=True)\n                loss=(F.binary_cross_entropy_with_logits(logits.float(),smoothed,reduction=\'none\',pos_weight=positive)*weight).sum()/weight.sum().clamp_min(1)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            before=scaler.get_scale(); scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),config[\'gradient_clip\'])\n            scaler.step(optimizer); scaler.update()\n            overflow=scaler.get_scale()<before\n            accumulated[\'loss\']+=float(loss.detach()); accumulated[\'batches\']+=1\n            if overflow:\n                accumulated[\'overflows\']+=1; consecutive_overflows+=1\n                if consecutive_overflows>=10:\n                    raise FloatingPointError(\'Ten consecutive FP16 gradient overflows\')\n            else:\n                consecutive_overflows=0; accumulated[\'updates\']+=1; scheduler.step()\n            next_batch=batch_index+1\n            if next_batch%100==0:\n                checkpoint(epoch,next_batch)\n                print(json.dumps({\'arm\':arm,\'fold\':fold,\'epoch\':epoch+1,\'batch\':next_batch,\'of\':len(loader),\n                                  \'loss\':float(loss),\'peak_gpu_gib\':torch.cuda.max_memory_allocated(device)/2**30}),flush=True)\n        if accumulated[\'overflows\']>max(10,.05*accumulated[\'batches\']):\n            raise FloatingPointError(\'More than5% of updates overflowed; refusing degraded model\')\n        ix,probs=predict(model,corpus,va,values,config,fold,device)\n        score,by_target=weak_scores(y[ix],probs)\n        record={\'epoch\':epoch+1,\'weak_macro_auc\':score,\'weak_per_target_auc\':by_target,\n                \'loss\':accumulated[\'loss\']/max(1,accumulated[\'batches\']),\n                \'seconds_this_session\':time.time()-epoch_started,\'updates\':accumulated[\'updates\'],\n                \'overflows\':accumulated[\'overflows\']}\n        history.append(record)\n        if score>best:\n            best=score\n            save_checkpoint(model,out/(prefix+\'_best.pt\'),contract=contract,extras={\'epoch\':epoch+1,\'weak_macro_auc\':score,\n                            \'train_prevalence\':prevalence.tolist()})\n            frame=pd.DataFrame(probs,columns=TARGETS); frame.insert(0,UID,corpus.ids[ix]);frame.to_csv(out/(prefix+\'_validation.csv\'),index=False)\n        epoch+=1; next_batch=0; accumulated={\'loss\':0.,\'batches\':0,\'overflows\':0,\'updates\':0}\n        checkpoint(epoch,0)\n        atomic_json({\'status\':\'complete\' if epoch==config[\'epochs\'] else \'running\',\'contract\':contract,\n                     \'completed_epochs\':epoch,\'best_weak_auc\':best,\'history\':history,\n                     \'peak_gpu_allocated_bytes\':torch.cuda.max_memory_allocated(device),\'seconds_this_session\':time.time()-started},status_path)\n        print(\'EPOCH_COMPLETE\',prefix,json.dumps(record),flush=True)\n        if epoch<config[\'epochs\'] and time.time()+record[\'seconds_this_session\']*1.2>args.deadline-reserve:\n            atomic_json({\'status\':\'partial\',\'contract\':contract,\'completed_epochs\':epoch,\'next_batch\':0,\n                         \'best_weak_auc\':best,\'history\':history,\'reason\':\'Insufficient budget for another complete epoch\'},status_path)\n            return\n\n\ndef completed_pairs(out,config):\n    result=[]\n    for fold in range(config[\'n_folds\']):\n        statuses=[]\n        for arm in (\'uniform\',\'spatial\'):\n            p=out/f\'{arm}_fold{fold}_status.json\'\n            statuses.append(json.loads(p.read_text()) if p.exists() else {\'status\':\'missing\'})\n        if all(s[\'status\']==\'complete\' and s[\'completed_epochs\']==config[\'epochs\'] for s in statuses):\n            result.append(fold)\n    return result\n\n\n@torch.inference_mode()\ndef holdout_once(plan,out,init_path):\n    marker=out/\'holdout_once_started.json\'\n    if marker.exists():\n        raise RuntimeError(\'Holdout already opened; no repeat selection loop is allowed\')\n    config=plan[\'config\']; complete=completed_pairs(out,config)\n    if len(complete)!=config[\'n_folds\']:\n        raise ValueError(\'All paired folds must complete before the locked holdout is opened\')\n    # Claim the once-only gate atomically before exposing held-out scores.\n    with marker.open(\'x\') as f:\n        json.dump({\'config\':config,\'pairs\':complete,\'selection_frozen\':True},f,indent=2)\n    corpus=Corpus(plan[\'corpus_paths\']); values=np.load(out/\'targets.npz\',allow_pickle=False)\n    rows=np.flatnonzero(values[\'roles\']==\'holdout\'); y=values[\'targets\'][rows]\n    device=torch.device(\'cuda\',0); results={}; predictions={}\n    for arm in (\'uniform\',\'spatial\'):\n        family=[]\n        for fold in complete:\n            status=json.loads((out/f\'{arm}_fold{fold}_status.json\').read_text())\n            model=initialized_model(config,arm,init_path,config[\'seed\']+fold*997).to(device)\n            checkpoint=out/f\'{arm}_fold{fold}_best.pt\'\n            load_checkpoint(model,checkpoint,contract=status[\'contract\'])\n            ix,p=predict(model,corpus,rows,values,config,fold,device)\n            order={int(r):i for i,r in enumerate(ix)}\n            family.append(p[[order[int(r)] for r in rows]])\n            del model; torch.cuda.empty_cache()\n        mean=np.mean(family,axis=0); predictions[arm]=mean\n        score,by_target=weak_scores(y,mean)\n        results[arm]={\'weak_macro_auc\':score,\'weak_per_target_auc\':by_target,\n                      \'checkpoints\':[{ \'file\':f\'{arm}_fold{fold}_best.pt\',\'sha256\':sha256(out/f\'{arm}_fold{fold}_best.pt\')} for fold in complete]}\n        frame=pd.DataFrame(mean,columns=TARGETS); frame.insert(0,UID,corpus.ids[rows]);frame.to_csv(out/f\'{arm}_holdout_predictions.csv\',index=False)\n    groups=values[\'groups\'][rows]; unique=np.unique(groups); rng=np.random.default_rng(config[\'seed\']+73)\n    bootstrap=[]\n    # Cluster bootstrap respects the same report/known-relation isolation unit.\n    for _ in range(1000):\n        selected=rng.choice(unique,len(unique),replace=True)\n        indices=np.concatenate([np.flatnonzero(groups==g) for g in selected])\n        try:\n            baseline,_=weak_scores(y[indices],predictions[\'uniform\'][indices])\n            spatial,_=weak_scores(y[indices],predictions[\'spatial\'][indices])\n        except ValueError:\n            continue\n        bootstrap.append(spatial-baseline)\n    delta=results[\'spatial\'][\'weak_macro_auc\']-results[\'uniform\'][\'weak_macro_auc\']\n    receipt={\'schema\':\'rsna_spatial_ablation_holdout_v1\',\'evaluation_count\':1,\'weak_label_proxy_not_expert_accuracy\':True,\n             \'gold_used_for_gradient_selection_or_this_evaluation\':False,\'pairs\':complete,\'rows\':len(rows),\n             \'results\':results,\'spatial_minus_uniform_weak_macro_auc\':delta,\n             \'group_bootstrap_replicates\':len(bootstrap),\n             \'group_bootstrap_delta_95pct\':None if not bootstrap else np.quantile(bootstrap,[.025,.975]).tolist(),\n             \'promotion_rule\':\'Research evidence only: no automatic production promotion or promised Kaggle score.\'}\n    atomic_json(receipt,out/\'holdout_comparison.json\')\n    print(\'HOLDOUT_ONCE_COMPLETE\',json.dumps(receipt),flush=True)\n\n\ndef launch(args):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is required to train; CPU CI checks architecture/contracts only\')\n    config=dict(DEFAULT_CONFIG)\n    if args.config:\n        config.update(json.loads(Path(args.config).read_text()))\n    if args.full_volume_sha256:\n        config[\'full_volume_sha256\']=True\n    if args.init_sha256:\n        config[\'initialization_sha256\']=args.init_sha256\n    if args.corpus_paths:\n        paths=json.loads(Path(args.corpus_paths).read_text())\n        if not args.train_csv or not args.teacher_csv:\n            raise ValueError(\'Explicit corpus paths require train.csv and weak-label CSV paths\')\n        train_csv,teachers=args.train_csv,args.teacher_csv\n    else:\n        paths,train_csv,teachers=discover_inputs(args.input_root)\n        train_csv=args.train_csv or train_csv; teachers=args.teacher_csv or teachers\n    if sha256(args.init)!=config[\'initialization_sha256\']:\n        raise ValueError(\'Initial ImageNet checkpoint hash differs\')\n    out=Path(args.out); out.mkdir(parents=True,exist_ok=True)\n    plan=make_plan(config,out,paths=paths,train_csv=train_csv,teacher_csvs=teachers,\n                   relations_csv=args.relations_csv,excluded_csv=args.excluded_csv)\n    deadline=time.time()+config[\'max_hours\']*3600\n    atomic_json({\'status\':\'running\',\'mode\':args.mode,\'packages\':versions(),\'config\':config,\n                 \'input_initialization_sha256\':sha256(args.init),\'gpus\':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                 \'gold_used_for_gradient_or_selection\':False,\'heldout_used_for_gradient_or_selection\':False},out/\'experiment_status.json\')\n    folds=[0] if args.mode==\'paired-first\' else list(range(config[\'n_folds\']))\n    for fold in folds:\n        if time.time()>deadline-config[\'reserve_minutes\']*60:\n            break\n        children=[]\n        for arm,gpu in [(\'uniform\',0),(\'spatial\',1 if torch.cuda.device_count()>1 else 0)]:\n            env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\',PYTHONUNBUFFERED=\'1\')\n            command=[sys.executable,str(Path(__file__).resolve()),\'--worker\',\'--plan\',str(out/\'plan.json\'),\n                     \'--out\',str(out),\'--init\',args.init,\'--arm\',arm,\'--fold\',str(fold),\'--deadline\',str(deadline)]\n            log=open(out/f\'{arm}_fold{fold}.log\',\'a\',encoding=\'utf-8\')\n            child=subprocess.Popen(command,env=env,stdout=log,stderr=subprocess.STDOUT)\n            children.append((child,log,arm))\n            if torch.cuda.device_count()==1:\n                remaining=max(1,deadline-time.time())\n                try:\n                    child.wait(timeout=remaining)\n                except subprocess.TimeoutExpired:\n                    child.terminate(); child.wait(timeout=20)\n        # Parent provides a hard deadline even if a driver/library hangs.\n        while any(p.poll() is None for p,_,_ in children):\n            if time.time()>deadline:\n                for p,_,_ in children:\n                    if p.poll() is None:\n                        p.terminate()\n                break\n            print(\'PAIR_RUNNING\',fold,{arm:p.poll() for p,_,arm in children},flush=True)\n            time.sleep(20)\n        failures=[]\n        for p,log,arm in children:\n            try:\n                p.wait(timeout=20)\n            except subprocess.TimeoutExpired:\n                p.kill(); p.wait()\n            log.close()\n            if p.returncode:\n                failures.append({\'arm\':arm,\'fold\':fold,\'exit_code\':p.returncode})\n        complete=completed_pairs(out,config)\n        if failures or fold not in complete:\n            atomic_json({\'status\':\'partial_or_failed\',\'completed_pairs\':complete,\'failures\':failures,\n                         \'partial_models_deployable\':False,\'training_receipts\':str(out)},out/\'experiment_status.json\')\n            print(\'PAIR_NOT_COMPLETE\',fold,\'See arm logs/status; resume the same output directory.\',flush=True)\n            return\n        print(\'PAIR_COMPLETE\',fold,flush=True)\n    complete=completed_pairs(out,config)\n    checkpoints=[]\n    for fold in complete:\n        for arm in (\'uniform\',\'spatial\'):\n            p=out/f\'{arm}_fold{fold}_best.pt\'\n            checkpoints.append({\'arm\':arm,\'fold\':fold,\'path\':str(p),\'sha256\':sha256(p)})\n    _status=\'all_pairs_complete\' if len(complete)==config[\'n_folds\'] else (\'paired_first_complete\' if complete else \'no_completed_pairs\')\n    atomic_json({\'status\':_status,\n                 \'completed_pairs\':complete,\'planned_pairs\':config[\'n_folds\'],\'config\':config,\'checkpoints\':checkpoints,\n                 \'partial_models_deployable\':False,\'gold_used_for_gradient_or_selection\':False,\n                 \'heldout_opened\':False,\'validation_boundary\':plan[\'validation_boundary\']},out/\'experiment_status.json\')\n    if args.holdout_once:\n        # Keep the once-only gate unopened when this training session has too\n        # little time for a conservative full-family evaluation. A later run\n        # skips the complete pairs and receives a fresh evaluation budget.\n        if time.time() + max(3600,config[\'reserve_minutes\']*120) > deadline:\n            atomic_json({\'status\':\'deferred\',\'reason\':\'Insufficient remaining GPU budget; rerun --mode all --holdout-once with the same output\',\n                         \'holdout_opened\':False},out/\'holdout_deferred.json\')\n            print(\'HOLDOUT_DEFERRED; completed training preserved; once-only gate remains unopened\',flush=True)\n        else:\n            holdout_once(plan,out,args.init)\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--out\',required=True); p.add_argument(\'--init\',required=True)\n    p.add_argument(\'--mode\',choices=[\'paired-first\',\'all\'],default=\'paired-first\')\n    p.add_argument(\'--config\');p.add_argument(\'--input-root\',default=\'/kaggle/input\')\n    p.add_argument(\'--corpus-paths\');p.add_argument(\'--train-csv\');p.add_argument(\'--teacher-csv\',action=\'append\')\n    p.add_argument(\'--relations-csv\');p.add_argument(\'--excluded-csv\');p.add_argument(\'--init-sha256\')\n    p.add_argument(\'--full-volume-sha256\',action=\'store_true\');p.add_argument(\'--holdout-once\',action=\'store_true\')\n    p.add_argument(\'--worker\',action=\'store_true\');p.add_argument(\'--plan\');p.add_argument(\'--arm\',choices=[\'uniform\',\'spatial\'])\n    p.add_argument(\'--fold\',type=int);p.add_argument(\'--deadline\',type=float)\n    a=p.parse_args()\n    if a.worker:\n        if a.plan is None or a.arm is None or a.fold is None or a.deadline is None:\n            p.error(\'Worker requires --plan, --arm, --fold, --deadline\')\n        try:\n            train_arm(a)\n        except Exception as exc:\n            atomic_json({\'status\':\'failed\',\'arm\':a.arm,\'fold\':a.fold,\'error\':repr(exc),\n                         \'partial_models_deployable\':False,\'resume_checkpoint_may_exist\':True},\n                        Path(a.out)/f\'{a.arm}_fold{a.fold}_failure.json\')\n            raise\n    else:\n        launch(a)\n\n\nif __name__==\'__main__\':\n    main()\n'}
EXPECTED_SHA256={'src/rsna/spatial_mil.py': 'af00256574386af4578607ade078aee4ddf34fbb36985e4b311c5f8e2e9e59bb', 'src/rsna/cache_data.py': '793f534444d19859e3c391e72f2d17179900f889565b04fe4e093ba15aa47a9f', 'scripts/train_spatial_ablation.py': '98c8ea731b53ddb279fe0eb7e8ee2447384319ed890565178c54e6610687bac8'}
for name,text in EMBEDDED.items():
    path=CODE_ROOT/name;path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text)
    assert hashlib.sha256(path.read_bytes()).hexdigest()==EXPECTED_SHA256[name],name
    compile(text,str(path),'exec')
(CODE_ROOT/'src/rsna/__init__.py').write_text('"""Original RSNA spatial ablation package."""\n')
sys.path.insert(0,str(CODE_ROOT/'src'))
from rsna.cache_data import discover_inputs,find_dataset_file,sha256,DEFAULT_CONFIG
paths,train_csv,teachers=discover_inputs()
INIT=Path(find_dataset_file('rsna-knee-overnight-convnext-td','imagenet_initialization.pt'))
assert sha256(INIT)==DEFAULT_CONFIG['initialization_sha256'],'Wrong ImageNet initialization bytes'
print('OFFLINE_INPUTS_READY',{'corpus_files':paths,'train_csv':train_csv,'teachers':teachers,
                             'initialization':str(INIT),'source_sha256':EXPECTED_SHA256},flush=True)


In [ ]:
env=dict(os.environ,PYTHONPATH=str(CODE_ROOT/'src')+os.pathsep+os.environ.get('PYTHONPATH',''),
         OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',PYTHONUNBUFFERED='1')
command=[sys.executable,'-u',str(CODE_ROOT/'scripts/train_spatial_ablation.py'),
         '--out',str(OUTPUT),'--init',str(INIT),'--mode',MODE]
if RELATIONS_CSV:command+=['--relations-csv',RELATIONS_CSV]
if EXCLUDED_CSV:command+=['--excluded-csv',EXCLUDED_CSV]
if EVALUATE_HOLDOUT_ONCE:command+=['--holdout-once']
started=time.time()
rc=subprocess.run(command,env=env,timeout=int(DEFAULT_CONFIG['max_hours']*3600+300)).returncode
assert rc==0,'Training failed: inspect per-arm logs/status before resuming'
status=json.loads((OUTPUT/'experiment_status.json').read_text())
print('ACTUAL_TRAINING_STATUS',json.dumps(status),flush=True)
print('Notebook elapsed hours:',round((time.time()-started)/3600,3),flush=True)
